In [29]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [4]:
df.drop_duplicates(inplace = True)

In [5]:
df.shape

(49582, 2)

# Pre-processing

## 1. Converting to lowercase

In [6]:
df["review"] = df["review"].str.lower()

## 2. Removing the URL's

In [7]:
import re

def remove_urls(text):
    text = re.sub(r"http\S+", "", text)
    return text

df["review"] = df["review"].apply(remove_urls)

## 3. Removing punctuations

In [8]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

## 4. Removing HTML

In [9]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [10]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,negative


## 5. Removing stopwords

In [11]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tanga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tanga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tanga\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [13]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("English")

    for w in tokens:
        if w in stop_words:
            text = text.replace(w, "")
    
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [14]:
df

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive
...,...,...
49995,thought ths move dd rght good job t wsnt s ...,positive
49996,bd plot bd dlogue bd ctng dotc drectng nnoyng...,negative
49997,ctholc tught n prochl elentry schools nuns...,negative
49998,im gog dgree previous comnt side mlt ...,negative


## 6. Stemming

In [15]:
from nltk.stem import PorterStemmer

In [16]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_tokens = ps.stem(token)
        stemmed_words.append(stemmed_tokens)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [17]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


## 7. Encoding

In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [19]:
y = df["sentiment"]

In [20]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

## 8. Vectorization

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])

## Dataset & DataLoader

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [23]:
X_train.shape

(39665, 5000)

In [24]:
X_test.shape

(9917, 5000)

In [27]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [35]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(), 
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(), 
    torch.from_numpy(y_test.values).float()
)

In [36]:
train_loader = DataLoader(train_set, shuffle = True, batch_size = 64)
test_loader = DataLoader(test_set, shuffle = True, batch_size = 64)

## Build our RNN

In [41]:
import torch.nn as nn
import torch.optim as optim

In [47]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size = 128, num_layers = 1):
        super().__init__()
    
        self.hidden_size = hidden_size
        self.num_layers = num_layers
    
        #RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)
    
        #Fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])

        return out

In [48]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [49]:
epochs = 10

for e in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)

        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {e+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.3290204405784607
epoch = 2/10 and loss = 0.35178086161613464
epoch = 3/10 and loss = 0.2688819468021393
epoch = 4/10 and loss = 0.2027754932641983
epoch = 5/10 and loss = 0.26158684492111206
epoch = 6/10 and loss = 0.22658810019493103
epoch = 7/10 and loss = 0.2716083228588104
epoch = 8/10 and loss = 0.27654916048049927
epoch = 9/10 and loss = 0.19316576421260834
epoch = 10/10 and loss = 0.35608765482902527


In [52]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

print(f"accuracy = {correct_vals/total_vals*100}")

accuracy = 85.73157204799838
